In [ ]:
!pip install sentence-transformers pandas numpy scipy openpyxl

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from scipy.optimize import linear_sum_assignment

In [ ]:
HUMAN_FOLDER = Path(r"C:\Users\Luiza\OneDrive\Desktop\MA THESIS\STORAL_prompt_versions\Prompt_v1\human_annotations")
LLM_FOLDER = Path(r"C:\Users\Luiza\OneDrive\Desktop\MA THESIS\STORAL_prompt_versions\Prompt_v1\llm_outputs")
OUTPUT_FOLDER = Path(r"C:\Users\Luiza\OneDrive\Desktop\MA THESIS\STORAL_prompt_versions\Prompt_v1\SBERT_eval_LASTPROMPT")

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

# helper fucntions for human Inception files

In [ ]:
IMPLICIT_LABELS = {
    "MCIMP": ("MC-IMP", "MCIMPtext"),
    "IClaim": ("C-IMP", "IClaimtext"),
    "PIMP": ("P-IMP", "PIMPtext"),
}

def extract_storal_number(filename):
    match = re.search(r"STORAL_(\d+)", filename)
    return f"STORAL_{match.group(1)}" if match else None


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def extract_human_implicit_annotations(human_json_path):
    data = load_json(human_json_path)
    
    sofa_text = ""
    for fs in data.get("%FEATURE_STRUCTURES", []):
        if fs.get("%TYPE") == "uima.cas.Sofa":
            sofa_text = fs.get("sofaString", "")
            break
    
    annotations = []
    
    for fs in data.get("%FEATURE_STRUCTURES", []):
        if fs.get("%TYPE") != "webanno.custom.Argumentation":
            continue
        
        begin = fs.get("begin")
        end = fs.get("end")
        supporting_span = sofa_text[begin:end] if begin is not None and end is not None else ""
        
        for flag_name, (label, text_field) in IMPLICIT_LABELS.items():
            if fs.get(flag_name) is True:
                explicit = fs.get(text_field, "")
                if explicit:
                    annotations.append({
                        "label": label,
                        "supporting_span": supporting_span,
                        "explicit_formulation": explicit,
                        "source": "human"
                    })
    
    return annotations

# helper functions for LLM files

In [ ]:
def normalize_llm_label(label):
    label = label.strip().upper()
    label = label.replace("_", "-")
    
    mapping = {
        "MCIMP": "MC-IMP",
        "MC-IMP": "MC-IMP",
        "CIMP": "C-IMP",
        "C-IMP": "C-IMP",
        "PIMP": "P-IMP",
        "P-IMP": "P-IMP",
    }
    
    return mapping.get(label, label)


def extract_llm_implicit_annotations(llm_json_path):
    data = load_json(llm_json_path)
    raw_annotations = data.get("annotations", [])
    
    annotations = []
    
    for ann in raw_annotations:
        label = normalize_llm_label(ann.get("label", ""))
        
        if label not in {"MC-IMP", "C-IMP", "P-IMP"}:
            continue
        
        annotations.append({
            "label": label,
            "supporting_span": ann.get("supporting_span", ann.get("span", ann.get("text", ""))),
            "explicit_formulation": ann.get("explicit_formulation", ann.get("explicit", "")),
            "source": "llm"
        })
    
    return annotations

# Load SBERT model

In [ ]:
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# semnatic similarity function

In [ ]:
def cosine_similarity(text1, text2):
    if not text1 or not text2:
        return np.nan
    
    embeddings = model.encode([text1, text2], normalize_embeddings=True)
    return float(np.dot(embeddings[0], embeddings[1]))

# Token overlap for supporting spans

In [ ]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


def token_overlap_f1(text1, text2):
    tokens1 = tokenize(text1)
    tokens2 = tokenize(text2)
    
    if not tokens1 or not tokens2:
        return np.nan
    
    set1 = set(tokens1)
    set2 = set(tokens2)
    
    overlap = len(set1 & set2)
    
    precision = overlap / len(set2)
    recall = overlap / len(set1)
    
    if precision + recall == 0:
        return 0.0
    
    return 2 * precision * recall / (precision + recall)

# Match human and LLM annotations by label

In [ ]:
def compare_annotations(human_annotations, llm_annotations, document_id, model_name):
    rows = []
    
    labels = sorted({"MC-IMP", "C-IMP", "P-IMP"})
    
    for label in labels:
        human_label_anns = [a for a in human_annotations if a["label"] == label]
        llm_label_anns = [a for a in llm_annotations if a["label"] == label]
        
        if not human_label_anns and not llm_label_anns:
            continue
        
        if human_label_anns and not llm_label_anns:
            for h in human_label_anns:
                rows.append({
                    "document_id": document_id,
                    "model": model_name,
                    "label": label,
                    "human_explicit": h["explicit_formulation"],
                    "llm_explicit": "",
                    "semantic_similarity": np.nan,
                    "supporting_span_f1": np.nan,
                    "match_status": "missed_by_llm"
                })
            continue
        
        if llm_label_anns and not human_label_anns:
            for l in llm_label_anns:
                rows.append({
                    "document_id": document_id,
                    "model": model_name,
                    "label": label,
                    "human_explicit": "",
                    "llm_explicit": l["explicit_formulation"],
                    "semantic_similarity": np.nan,
                    "supporting_span_f1": np.nan,
                    "match_status": "extra_llm_annotation"
                })
            continue
        
        sim_matrix = np.zeros((len(human_label_anns), len(llm_label_anns)))
        
        for i, h in enumerate(human_label_anns):
            for j, l in enumerate(llm_label_anns):
                sim_matrix[i, j] = cosine_similarity(
                    h["explicit_formulation"],
                    l["explicit_formulation"]
                )
        
        row_ind, col_ind = linear_sum_assignment(-sim_matrix)
        matched_h = set()
        matched_l = set()
        
        for i, j in zip(row_ind, col_ind):
            h = human_label_anns[i]
            l = llm_label_anns[j]
            matched_h.add(i)
            matched_l.add(j)
            
            rows.append({
                "document_id": document_id,
                "model": model_name,
                "label": label,
                "human_explicit": h["explicit_formulation"],
                "llm_explicit": l["explicit_formulation"],
                "semantic_similarity": sim_matrix[i, j],
                "supporting_span_f1": token_overlap_f1(
                    h["supporting_span"],
                    l["supporting_span"]
                ),
                "human_span": h["supporting_span"],
                "llm_span": l["supporting_span"],
                "match_status": "matched"
            })
        
        for i, h in enumerate(human_label_anns):
            if i not in matched_h:
                rows.append({
                    "document_id": document_id,
                    "model": model_name,
                    "label": label,
                    "human_explicit": h["explicit_formulation"],
                    "llm_explicit": "",
                    "semantic_similarity": np.nan,
                    "supporting_span_f1": np.nan,
                    "human_span": h["supporting_span"],
                    "llm_span": "",
                    "match_status": "missed_by_llm"
                })
        
        for j, l in enumerate(llm_label_anns):
            if j not in matched_l:
                rows.append({
                    "document_id": document_id,
                    "model": model_name,
                    "label": label,
                    "human_explicit": "",
                    "llm_explicit": l["explicit_formulation"],
                    "semantic_similarity": np.nan,
                    "supporting_span_f1": np.nan,
                    "human_span": "",
                    "llm_span": l["supporting_span"],
                    "match_status": "extra_llm_annotation"
                })
    
    return rows

# Detect model name from filename

In [ ]:
def detect_model_name(filename):
    filename = filename.lower()
    
    if "gemma" in filename:
        return "gemma"
    if "mistral" in filename:
        return "mistral"
    if "qwen" in filename:
        return "qwen"
    
    return "unknown"

# Run evaluation for all STORAL files

In [ ]:
all_rows = []

human_files = list(HUMAN_FOLDER.glob("*.json"))
llm_files = list(LLM_FOLDER.glob("*.json"))

for human_file in human_files:
    document_id = extract_storal_number(human_file.name)
    
    if document_id is None:
        continue
    
    human_annotations = extract_human_implicit_annotations(human_file)
    
    matching_llm_files = [
        f for f in llm_files
        if document_id in f.name and "implicit" in f.name.lower()
    ]
    
    print(f"\nProcessing {document_id}")
    print(f"Human annotations: {len(human_annotations)}")
    print(f"LLM files found: {len(matching_llm_files)}")
    
    for llm_file in matching_llm_files:
        model_name = detect_model_name(llm_file.name)
        llm_annotations = extract_llm_implicit_annotations(llm_file)
        
        print(f"  {model_name}: {len(llm_annotations)} annotations")
        
        rows = compare_annotations(
            human_annotations=human_annotations,
            llm_annotations=llm_annotations,
            document_id=document_id,
            model_name=model_name
        )
        
        all_rows.extend(rows)

results_df = pd.DataFrame(all_rows)
results_df

In [ ]:
detailed_output_path = OUTPUT_FOLDER / "storal_implicit_semantic_similarity_detailed_LASTPROMPT.xlsx"
results_df.to_excel(detailed_output_path, index=False)

print(f"Saved detailed results to: {detailed_output_path}")

# Summary by model and label

In [ ]:
summary_by_model_label = (
    results_df[results_df["match_status"] == "matched"]
    .groupby(["model", "label"])
    .agg(
        avg_semantic_similarity=("semantic_similarity", "mean"),
        avg_supporting_span_f1=("supporting_span_f1", "mean"),
        number_of_matches=("semantic_similarity", "count")
    )
    .reset_index()
)

summary_by_model_label

# overall summary by SBERT

In [ ]:
summary_by_model = (
    results_df[results_df["match_status"] == "matched"]
    .groupby("model")
    .agg(
        avg_semantic_similarity=("semantic_similarity", "mean"),
        avg_supporting_span_f1=("supporting_span_f1", "mean"),
        number_of_matches=("semantic_similarity", "count")
    )
    .reset_index()
    .sort_values("avg_semantic_similarity", ascending=False)
)

summary_by_model

# save summaries

In [ ]:
summary_path = OUTPUT_FOLDER / "storal_implicit_semantic_similarity_summary.xlsx"

with pd.ExcelWriter(summary_path) as writer:
    results_df.to_excel(writer, sheet_name="detailed_results", index=False)
    summary_by_model_label.to_excel(writer, sheet_name="by_model_label", index=False)
    summary_by_model.to_excel(writer, sheet_name="by_model", index=False)

print(f"Saved summary file to: {summary_path}")

# interpret scores

In [ ]:
def interpret_similarity(score):
    if pd.isna(score):
        return "no match"
    elif score >= 0.90:
        return "near-equivalent"
    elif score >= 0.75:
        return "strong similarity"
    elif score >= 0.60:
        return "partial similarity"
    else:
        return "weak similarity"


results_df["semantic_interpretation"] = results_df["semantic_similarity"].apply(interpret_similarity)
results_df[["document_id", "model", "label", "semantic_similarity", "semantic_interpretation", "match_status"]]

In [ ]:
def interpret_similarity(score):
    if pd.isna(score):
        return "no match"
    elif score >= 0.80:
        return "very strong alignment"
    elif score >= 0.65:
        return "strong alignment"
    elif score >= 0.50:
        return "moderate alignment"
    elif score >= 0.35:
        return "weak but meaningful alignment"
    else:
        return "poor alignment"

results_df["semantic_interpretation"] = results_df["semantic_similarity"].apply(interpret_similarity)
results_df[["document_id", "model", "label", "semantic_similarity", "semantic_interpretation", "match_status"]]

In [ ]:
# Add interpretation column
def interpret_similarity(score):
    if pd.isna(score):
        return "no match"
    elif score >= 0.80:
        return "very strong alignment"
    elif score >= 0.65:
        return "strong alignment"
    elif score >= 0.50:
        return "moderate alignment"
    elif score >= 0.35:
        return "weak but meaningful alignment"
    else:
        return "poor alignment"


results_df["semantic_interpretation"] = (
    results_df["semantic_similarity"]
    .apply(interpret_similarity)
)

# Create the table you want to inspect
interpretation_table = results_df[
    [
        "document_id",
        "model",
        "label",
        "semantic_similarity",
        "semantic_interpretation",
        "match_status"
    ]
]

# Display it in notebook
display(interpretation_table)

# Save locally
save_path = OUTPUT_FOLDER / "semantic_similarity_interpretation_table.xlsx"

interpretation_table.to_excel(save_path, index=False)

print(f"Saved table to:\n{save_path}")

# BERTscore

In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
!pip install bert-score pandas numpy scipy openpyxl transformers

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np
from scipy.optimize import linear_sum_assignment
from bert_score import BERTScorer

## set folders

In [ ]:
HUMAN_FOLDER = Path(r"C:\Users\Luiza\OneDrive\Desktop\MA THESIS\STORAL_prompt_versions\Prompt_v1\human_annotations")
LLM_FOLDER = Path(r"C:\Users\Luiza\OneDrive\Desktop\MA THESIS\STORAL_prompt_versions\Prompt_v1\llm_outputs")
OUTPUT_FOLDER = Path(r"C:\Users\Luiza\OneDrive\Desktop\MA THESIS\STORAL_prompt_versions\Prompt_v1\BERTScore_eval_LASTPROMPT")

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

## Load BERTscore model


In [ ]:
from bert_score import BERTScorer

bert_scorer = BERTScorer(
    model_type="roberta-large",
    lang="en",
    rescale_with_baseline=True
)

# Fix tokenizer overflow bug
bert_scorer._tokenizer.model_max_length = 512

## basic helper functions

In [ ]:
IMPLICIT_LABELS = {
    "MCIMP": ("MC-IMP", "MCIMPtext"),
    "IClaim": ("C-IMP", "IClaimtext"),
    "PIMP": ("P-IMP", "PIMPtext"),
}

def extract_storal_number(filename):
    match = re.search(r"STORAL_(\d+)", filename)
    return f"STORAL_{match.group(1)}" if match else None


def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

## extract human INCEPTION annotation

In [ ]:
def extract_human_implicit_annotations(human_json_path):
    data = load_json(human_json_path)

    sofa_text = ""
    for fs in data.get("%FEATURE_STRUCTURES", []):
        if fs.get("%TYPE") == "uima.cas.Sofa":
            sofa_text = fs.get("sofaString", "")
            break

    annotations = []

    for fs in data.get("%FEATURE_STRUCTURES", []):
        if fs.get("%TYPE") != "webanno.custom.Argumentation":
            continue

        begin = fs.get("begin")
        end = fs.get("end")
        supporting_span = sofa_text[begin:end] if begin is not None and end is not None else ""

        for flag_name, (label, text_field) in IMPLICIT_LABELS.items():
            if fs.get(flag_name) is True:
                explicit = fs.get(text_field, "")

                if explicit:
                    annotations.append({
                        "label": label,
                        "supporting_span": supporting_span,
                        "explicit_formulation": explicit,
                        "source": "human"
                    })

    return annotations

## Extract LLM annotations

In [ ]:
def normalize_llm_label(label):
    label = label.strip().upper().replace("_", "-")

    mapping = {
        "MCIMP": "MC-IMP",
        "MC-IMP": "MC-IMP",
        "CIMP": "C-IMP",
        "C-IMP": "C-IMP",
        "PIMP": "P-IMP",
        "P-IMP": "P-IMP",
    }

    return mapping.get(label, label)


def extract_llm_implicit_annotations(llm_json_path):
    data = load_json(llm_json_path)
    raw_annotations = data.get("annotations", [])

    annotations = []

    for ann in raw_annotations:
        label = normalize_llm_label(ann.get("label", ""))

        if label not in {"MC-IMP", "C-IMP", "P-IMP"}:
            continue

        annotations.append({
            "label": label,
            "supporting_span": ann.get("supporting_span", ann.get("span", ann.get("text", ""))),
            "explicit_formulation": ann.get("explicit_formulation", ann.get("explicit", "")),
            "source": "llm"
        })

    return annotations

## BERTScore function

In [ ]:
def compute_bertscore(reference_text, candidate_text):
    """
    reference_text = human explicit formulation
    candidate_text = LLM explicit formulation
    """

    if not reference_text or not candidate_text:
        return {
            "bertscore_precision": np.nan,
            "bertscore_recall": np.nan,
            "bertscore_f1": np.nan
        }

    reference_text = str(reference_text).strip()
    candidate_text = str(candidate_text).strip()

    if reference_text == "" or candidate_text == "":
        return {
            "bertscore_precision": np.nan,
            "bertscore_recall": np.nan,
            "bertscore_f1": np.nan
        }

    try:
        P, R, F1 = bert_scorer.score(
            [candidate_text],
            [reference_text],
            batch_size=1
        )

        return {
            "bertscore_precision": float(P[0]),
            "bertscore_recall": float(R[0]),
            "bertscore_f1": float(F1[0])
        }

    except OverflowError as e:
        print("OverflowError for pair:")
        print("REFERENCE:", reference_text)
        print("CANDIDATE:", candidate_text)
        print("Error:", e)

        return {
            "bertscore_precision": np.nan,
            "bertscore_recall": np.nan,
            "bertscore_f1": np.nan
        }

## Suppporting span F1

In [ ]:
def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())


def token_overlap_f1(text1, text2):
    tokens1 = tokenize(text1)
    tokens2 = tokenize(text2)

    if not tokens1 or not tokens2:
        return np.nan

    set1 = set(tokens1)
    set2 = set(tokens2)

    overlap = len(set1 & set2)

    precision = overlap / len(set2)
    recall = overlap / len(set1)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)

## compare annotation using BERTScore

In [ ]:
def compare_annotations_bertscore(human_annotations, llm_annotations, document_id, model_name):
    rows = []

    labels = ["MC-IMP", "C-IMP", "P-IMP"]

    for label in labels:
        human_label_anns = [a for a in human_annotations if a["label"] == label]
        llm_label_anns = [a for a in llm_annotations if a["label"] == label]

        if not human_label_anns and not llm_label_anns:
            continue

        if human_label_anns and not llm_label_anns:
            for h in human_label_anns:
                rows.append({
                    "document_id": document_id,
                    "model": model_name,
                    "label": label,
                    "human_explicit": h["explicit_formulation"],
                    "llm_explicit": "",
                    "bertscore_precision": np.nan,
                    "bertscore_recall": np.nan,
                    "bertscore_f1": np.nan,
                    "supporting_span_f1": np.nan,
                    "match_status": "missed_by_llm"
                })
            continue

        if llm_label_anns and not human_label_anns:
            for l in llm_label_anns:
                rows.append({
                    "document_id": document_id,
                    "model": model_name,
                    "label": label,
                    "human_explicit": "",
                    "llm_explicit": l["explicit_formulation"],
                    "bertscore_precision": np.nan,
                    "bertscore_recall": np.nan,
                    "bertscore_f1": np.nan,
                    "supporting_span_f1": np.nan,
                    "match_status": "extra_llm_annotation"
                })
            continue

        score_matrix = np.zeros((len(human_label_anns), len(llm_label_anns)))

        bertscore_cache = {}

        for i, h in enumerate(human_label_anns):
            for j, l in enumerate(llm_label_anns):
                scores = compute_bertscore(
                    reference_text=h["explicit_formulation"],
                    candidate_text=l["explicit_formulation"]
                )

                bertscore_cache[(i, j)] = scores
                score_matrix[i, j] = scores["bertscore_f1"]

        row_ind, col_ind = linear_sum_assignment(-score_matrix)

        matched_h = set()
        matched_l = set()

        for i, j in zip(row_ind, col_ind):
            h = human_label_anns[i]
            l = llm_label_anns[j]
            scores = bertscore_cache[(i, j)]

            matched_h.add(i)
            matched_l.add(j)

            rows.append({
                "document_id": document_id,
                "model": model_name,
                "label": label,
                "human_explicit": h["explicit_formulation"],
                "llm_explicit": l["explicit_formulation"],
                "bertscore_precision": scores["bertscore_precision"],
                "bertscore_recall": scores["bertscore_recall"],
                "bertscore_f1": scores["bertscore_f1"],
                "supporting_span_f1": token_overlap_f1(
                    h["supporting_span"],
                    l["supporting_span"]
                ),
                "human_span": h["supporting_span"],
                "llm_span": l["supporting_span"],
                "match_status": "matched"
            })

        for i, h in enumerate(human_label_anns):
            if i not in matched_h:
                rows.append({
                    "document_id": document_id,
                    "model": model_name,
                    "label": label,
                    "human_explicit": h["explicit_formulation"],
                    "llm_explicit": "",
                    "bertscore_precision": np.nan,
                    "bertscore_recall": np.nan,
                    "bertscore_f1": np.nan,
                    "supporting_span_f1": np.nan,
                    "human_span": h["supporting_span"],
                    "llm_span": "",
                    "match_status": "missed_by_llm"
                })

        for j, l in enumerate(llm_label_anns):
            if j not in matched_l:
                rows.append({
                    "document_id": document_id,
                    "model": model_name,
                    "label": label,
                    "human_explicit": "",
                    "llm_explicit": l["explicit_formulation"],
                    "bertscore_precision": np.nan,
                    "bertscore_recall": np.nan,
                    "bertscore_f1": np.nan,
                    "supporting_span_f1": np.nan,
                    "human_span": "",
                    "llm_span": l["supporting_span"],
                    "match_status": "extra_llm_annotation"
                })

    return rows

## detect model name

In [ ]:
def detect_model_name(filename):
    filename = filename.lower()

    if "gemma" in filename:
        return "gemma"
    if "mistral" in filename:
        return "mistral"
    if "qwen" in filename:
        return "qwen"

    return "unknown"

## run BERTScore evalation for ALL STORAL files

In [ ]:
all_bertscore_rows = []

human_files = list(HUMAN_FOLDER.glob("*.json"))
llm_files = list(LLM_FOLDER.glob("*.json"))

for human_file in human_files:
    document_id = extract_storal_number(human_file.name)

    if document_id is None:
        continue

    human_annotations = extract_human_implicit_annotations(human_file)

    matching_llm_files = [
        f for f in llm_files
        if document_id in f.name and "implicit" in f.name.lower()
    ]

    print(f"\nProcessing {document_id}")
    print(f"Human annotations: {len(human_annotations)}")
    print(f"LLM files found: {len(matching_llm_files)}")

    for llm_file in matching_llm_files:
        model_name = detect_model_name(llm_file.name)
        llm_annotations = extract_llm_implicit_annotations(llm_file)

        print(f"  {model_name}: {len(llm_annotations)} annotations")

        rows = compare_annotations_bertscore(
            human_annotations=human_annotations,
            llm_annotations=llm_annotations,
            document_id=document_id,
            model_name=model_name
        )

        all_bertscore_rows.extend(rows)

bertscore_df = pd.DataFrame(all_bertscore_rows)
bertscore_df

## Interpret BERTScore

In [ ]:
def interpret_bertscore(score):
    if pd.isna(score):
        return "no match"
    elif score >= 0.80:
        return "very strong semantic alignment"
    elif score >= 0.65:
        return "strong semantic alignment"
    elif score >= 0.50:
        return "moderate semantic alignment"
    elif score >= 0.35:
        return "weak but meaningful alignment"
    else:
        return "poor alignment"


bertscore_df["bertscore_interpretation"] = (
    bertscore_df["bertscore_f1"].apply(interpret_bertscore)
)

bertscore_df[
    [
        "document_id",
        "model",
        "label",
        "bertscore_precision",
        "bertscore_recall",
        "bertscore_f1",
        "bertscore_interpretation",
        "match_status"
    ]
]

## summary by model and label

In [ ]:
bertscore_summary_by_model_label = (
    bertscore_df[bertscore_df["match_status"] == "matched"]
    .groupby(["model", "label"])
    .agg(
        avg_bertscore_precision=("bertscore_precision", "mean"),
        avg_bertscore_recall=("bertscore_recall", "mean"),
        avg_bertscore_f1=("bertscore_f1", "mean"),
        avg_supporting_span_f1=("supporting_span_f1", "mean"),
        number_of_matches=("bertscore_f1", "count")
    )
    .reset_index()
)

bertscore_summary_by_model_label

## overall summary of BERTScore

In [ ]:
bertscore_summary_by_model = (
    bertscore_df[bertscore_df["match_status"] == "matched"]
    .groupby("model")
    .agg(
        avg_bertscore_precision=("bertscore_precision", "mean"),
        avg_bertscore_recall=("bertscore_recall", "mean"),
        avg_bertscore_f1=("bertscore_f1", "mean"),
        avg_supporting_span_f1=("supporting_span_f1", "mean"),
        number_of_matches=("bertscore_f1", "count")
    )
    .reset_index()
    .sort_values("avg_bertscore_f1", ascending=False)
)

bertscore_summary_by_model

## save everything locally

In [ ]:
output_path = OUTPUT_FOLDER_BERT / "storal_implicit_bertscore_results.xlsx"

with pd.ExcelWriter(output_path) as writer:
    bertscore_df.to_excel(writer, sheet_name="detailed_results", index=False)
    bertscore_summary_by_model_label.to_excel(writer, sheet_name="by_model_label", index=False)
    bertscore_summary_by_model.to_excel(writer, sheet_name="by_model", index=False)

print(f"Saved BERTScore results to:\n{output_path}")

## combine SBERT and BERTScore, side by side for easier visibility

In [ ]:
combined_df = results_df.merge(
    bertscore_df[
        [
            "document_id",
            "model",
            "label",
            "human_explicit",
            "llm_explicit",
            "bertscore_precision",
            "bertscore_recall",
            "bertscore_f1",
            "bertscore_interpretation"
        ]
    ],
    on=["document_id", "model", "label", "human_explicit", "llm_explicit"],
    how="outer"
)

combined_output_path = OUTPUT_FOLDER / "storal_implicit_sbert_and_bertscore_combined.xlsx"
combined_df.to_excel(combined_output_path, index=False)

print(f"Saved combined SBERT + BERTScore results to:\n{combined_output_path}")

In [ ]:
combined_df